In [0]:
# --- CELDA 1: SETUP ---
%pip install scikit-learn pandas --quiet
dbutils.library.restartPython()

In [0]:
# --- CELDA 2: RUTAS (Tu código robusto) ---
import sys, os
notebook_path = os.getcwd()
project_root = os.path.abspath(os.path.join(notebook_path, ".."))
FOLDER_NAME = "src" 

src_path = os.path.join(project_root, FOLDER_NAME)
if not os.path.exists(src_path): src_path = os.path.join(notebook_path, FOLDER_NAME)
if src_path not in sys.path: sys.path.append(src_path)


In [0]:
# --- CELDA 3: IMPORTS ---
# Importamos el archivo que acabamos de crear
from nombre_paquete.preprocessing import transformers
import pandas as pd

# Autoreload para desarrollo rápido
%load_ext autoreload
%autoreload 2


In [0]:
# --- CELDA 4: CARGAR DATOS (Bronze Layer) ---
table_input = "climate_data_raw"
print(f"📥 Leyendo datos desde: {table_input}...")

try:
    # Leemos la tabla Delta guardada en el Notebook 01
    df_raw = spark.table(table_input).toPandas()
    print(f"✅ Datos cargados exitosamente. Filas: {len(df_raw)}")
except Exception as e:
    print(f"❌ Error: La tabla '{table_input}' no existe. Ejecuta el Notebook 01 primero.")


In [0]:

# --- CELDA 5: EJECUTAR LIMPIEZA ---
# Llamamos a la función 1 de tu script
df_clean = transformers.preprocess_data(df_raw)

# Verificación rápida
print("Muestra de datos procesados:")
display(df_clean.head())

In [0]:
# --- CELDA 6: DIVISIÓN Y ESCALADO ---
# Definimos fecha de corte (asegúrate que tenga sentido con tus datos)
FECHA_CORTE = '2023-01-01' 

# Llamamos a la función 2 de tu script
train_df, test_df, scaler_model = transformers.split_and_scale(df_clean, cutoff_date=FECHA_CORTE)

print("-" * 30)
print(f"📊 Dimensiones Train: {train_df.shape}")
print(f"📊 Dimensiones Test:  {test_df.shape}")
print("-" * 30)


In [0]:
# Clean column names to remove invalid characters
def clean_column_names(df):
    df.columns = [
        col.strip()
        .replace(' ', '_')
        .replace(',', '')
        .replace(';', '')
        .replace('{', '')
        .replace('}', '')
        .replace('(', '')
        .replace(')', '')
        .replace('\n', '')
        .replace('\t', '')
        .replace('=', '')
        for col in df.columns
    ]
    return df

train_save = clean_column_names(train_df.reset_index())
test_save = clean_column_names(test_df.reset_index())

print("💾 Guardando tablas procesadas en el Data Lake...")

spark.createDataFrame(train_save)\
    .write.format("delta").mode("overwrite").saveAsTable("climate_train_silver")

spark.createDataFrame(test_save)\
    .write.format("delta").mode("overwrite").saveAsTable("climate_test_silver")

print("✅ Éxito: Tablas 'climate_train_silver' y 'climate_test_silver' creadas.")